# Market Capitalization Evolution

This notebook plots the historical market capitalization of **Meta**, **Google**, **Nvidia**, **Amazon**, and **Microsoft** from the `enterprise_metrics_history` table.

- Values are expressed in **millions of US dollars** (`usd_m`).
- The data source is the local SQLite database `database.db`.
- Points are displayed as stored in the database, without interpolating missing years.

## 1. Load and configure

The database path is searched from the current directory and its parents so the notebook works from VS Code, Jupyter, or `nbconvert`.

In [22]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


def find_database_path():
    """Find the project database from the notebook working directory."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for directory in candidates:
        database_path = directory / 'database.db'
        if database_path.exists():
            return database_path
    raise FileNotFoundError('database.db was not found from the current directory')


# Keep the chart reproducible by defining the tracked companies and their colors once.
DB_PATH = find_database_path()
TARGETS = ['Meta', 'Google', 'Nvidia', 'Amazon', 'Microsoft']
COLORS = {
    'Meta': '#1877F2',
    'Google': '#E45756',
    'Nvidia': '#76B900',
    'Amazon': '#FF9900',
    'Microsoft': '#00A4EF',
}

print(f'Database: {DB_PATH}')
print('Tracked companies:', ', '.join(TARGETS))

Database: c:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\database.db
Tracked companies: Meta, Google, Nvidia, Amazon, Microsoft


## 2. Extract the historical series

The join with `enterprises` keeps the entity names readable and avoids relying on numeric identifiers.

In [18]:
# Query only the controlled metric and unit used by the historical capitalization model.
query = f"""
    SELECT
        e.name AS enterprise,
        h.year,
        h.value AS capitalization_millions,
        h.unit,
        h.indicator
    FROM enterprise_metrics_history AS h
    JOIN enterprises AS e ON e.name = h.enterprise_name
    WHERE e.name IN ({', '.join('?' for _ in TARGETS)})
      AND h.indicator = 'capitalization'
      AND h.unit = 'usd_m'
    ORDER BY h.year, e.name
"""

with sqlite3.connect(DB_PATH) as connection:
    df = pd.read_sql_query(query, connection, params=TARGETS)

# Keep the raw database values in millions and derive billions only for chart readability.
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['capitalization_millions'] = pd.to_numeric(df['capitalization_millions'], errors='coerce')
df['capitalization_billions'] = df['capitalization_millions'] / 1_000
df = df.dropna(subset=['year', 'capitalization_millions']).copy()
df['year'] = df['year'].astype(int)

if df.empty:
    raise ValueError('No capitalization/usd_m data was found for the tracked companies')

missing_targets = sorted(set(TARGETS) - set(df['enterprise']))
if missing_targets:
    print('Warning: no series found for:', ', '.join(missing_targets))

display(df[['enterprise', 'year', 'capitalization_millions', 'capitalization_billions']].round(2))

,enterprise,year,capitalization_millions,capitalization_billions
0,Microsoft,1996,99400.0,99.40
1,Amazon,1997,440.0,0.44
2,Microsoft,1999,604400.0,604.40
3,Amazon,2000,5500.0,5.50
4,Microsoft,2000,230800.0,230.80
...,...,...,...,...
70,Nvidia,2025,2970000.0,2970.00
71,Amazon,2026,2860000.0,2860.00
72,Google,2026,4200000.0,4200.00
73,Meta,2026,1510000.0,1510.00


## 3. Evolution chart

The main view uses billions of US dollars on a logarithmic scale. Markers show the years documented in the database, hover labels provide exact values, and very thin dashed lines show each company's linear trend fitted in `log10` space.

In [ ]:
# Build one trace per company so gaps in the source data remain visible.
fig = go.Figure()

for enterprise in TARGETS:
    series = df[df['enterprise'] == enterprise].sort_values('year')
    if series.empty:
        continue

    fig.add_trace(go.Scatter(
        x=series['year'],
        y=series['capitalization_billions'],
        mode='lines+markers',
        name=enterprise,
        line={'color': COLORS[enterprise], 'width': 3},
        marker={'color': COLORS[enterprise], 'size': 8, 'line': {'color': '#FFFFFF', 'width': 1}},
        customdata=series[['capitalization_millions']].to_numpy(),
        hovertemplate='<b>%{fullData.name}</b><br>Year: %{x}<br>Market capitalization: %{y:.1f} B$<br>Exact value: %{customdata[0]:,.0f} M$<extra></extra>'
    ))

    # Fit log10(capitalization) against year, matching the logarithmic Y axis.
    if len(series) >= 2 and (series['capitalization_billions'] > 0).all():
        regression_coefficients = np.polyfit(
            series['year'],
            np.log10(series['capitalization_billions']),
            1
        )
        regression_years = np.linspace(series['year'].min(), series['year'].max(), 100)
        regression_values = 10 ** np.polyval(regression_coefficients, regression_years)

        fig.add_trace(go.Scatter(
            x=regression_years,
            y=regression_values,
            mode='lines',
            name=f'{enterprise} trend',
            legendgroup=enterprise,
            showlegend=False,
            line={'color': COLORS[enterprise], 'width': 1, 'dash': 'dot'},
            hoverinfo='skip'
        ))

# A logarithmic Y axis makes early and recent market-capitalization levels comparable.
fig.update_layout(
    title={
        'text': 'Meta, Google, Nvidia, Amazon and Microsoft: Market Capitalization Evolution',
        'x': 0.03,
        'xanchor': 'left',
        'font': {'size': 22, 'color': '#28241E'}
    },
    xaxis={
        'title': 'Year',
        'dtick': 2,
        'showgrid': False,
        'zeroline': False
    },
    yaxis={
        'title': 'Market capitalization (US$ billions, log scale)',
        'type': 'log',
        'dtick': 1,
        'exponentformat': 'SI',
        'showexponent': 'all',
        'gridcolor': '#E4DED5',
        'zeroline': False
    },
    template='plotly_white',
    hovermode='x unified',
    legend={'orientation': 'h', 'y': 1.08, 'x': 0, 'title': None},
    margin={'l': 75, 'r': 30, 't': 105, 'b': 65},
    height=620,
    paper_bgcolor='#FDFAF4',
    plot_bgcolor='#FDFAF4'
)

fig.show()

## 4. Data coverage check

This table summarizes the coverage of each series and makes missing years or differences in availability easy to identify.

In [20]:
# Summarize coverage and value ranges for a quick data-quality check.
summary = (
    df.groupby('enterprise', as_index=False)
      .agg(
          observations=('year', 'count'),
          first_year=('year', 'min'),
          last_year=('year', 'max'),
          min_capitalization_millions=('capitalization_millions', 'min'),
          max_capitalization_millions=('capitalization_millions', 'max')
      )
)

summary['min_capitalization_billions'] = summary['min_capitalization_millions'] / 1_000
summary['max_capitalization_billions'] = summary['max_capitalization_millions'] / 1_000

display(summary[['enterprise', 'observations', 'first_year', 'last_year', 'min_capitalization_billions', 'max_capitalization_billions']].round(2))

,enterprise,observations,first_year,last_year,min_capitalization_billions,max_capitalization_billions
0,Amazon,16,1997,2026,0.44,2860.0
1,Google,15,2004,2026,47.20,4200.0
2,Meta,15,2012,2026,63.10,1850.0
3,Microsoft,14,1996,2025,99.40,3620.0
4,Nvidia,15,2012,2026,7.06,5470.0
